# control_hub_v1

Schematic-only flow (v1). One subsystem per cell — pick part, attach math, check, render inline. Final cells connect, ERC, write `.kicad_sch`.

Doctrine: **load-first** — actuators / sensors / MCU lock before power rails. Math + thermal check per module before connect.

| cell | step |
|---|---|
| 1 | load Board |
| 2 | buck_3v3 (rail) — math + thermal check + show |
| 3 | mcu — pick + show |
| 4 | imu — pick + show |
| 5 | connect modules |
| 6 | check ERC + render full + write `.kicad_sch` |

## Cell 1 — load

Construct `Board` directly. No JSON — code is the source of truth.

In [ ]:
import hw_toolkit as hw

board = hw.Board("control_hub_v1")
board

## Cell 2 — buck_3v3 (rail)

Step-down: vbat 11.1V → 3.3V @ 0.5A. Math + thermal must pass before this cell completes.

In [ ]:
buck = board.module(
    id="buck_3v3", category="buck_converter",
    mpn="TPS54331DR", manufacturer="TI", package="SOIC-8",
    lcsc="C9865", price_usd=1.05, qty_per_board=1,
)
buck.attach(hw.calc.Buck(vin=11.1, vout=3.3, iout=0.5))
T = buck.math.thermal(rdson_mohm=80, theta_ja=40)
buck.check(T, label=f"thermal tj={T.tj_c:.1f}C margin={T.margin_c:.1f}C")
buck.show()

## Cell 3 — mcu

ESP32-S3-WROOM. Decoupling caps + boot/strapping resistors attached in eeschema after `write_kicad()`.

In [ ]:
mcu = board.module(
    id="mcu", category="mcu_module",
    mpn="ESP32-S3-WROOM-1-N16R8", manufacturer="Espressif", package="SMD-Module",
    lcsc="C2913202", price_usd=4.92, qty_per_board=1,
)
mcu.show()

## Cell 4 — imu

LSM6DSOX 6-axis IMU on I²C. Pull-ups (4.7k) attached in eeschema.

In [ ]:
imu = board.module(
    id="imu", category="imu_i2c",
    mpn="LSM6DSOXTR", manufacturer="ST", package="LGA-14",
    lcsc="C481766", price_usd=3.19, qty_per_board=1,
)
imu.show()

## Cell 5 — nets (buckets)

Declare each electrical net (power rail, bus) and drop the pins that join it. `net += "a.x", "b.y", ...` joins members in one line. Buckets scale better than pairwise `connect()` for multi-drop nets (power, I²C, SPI).

Star-expansion under the hood: an N-member net becomes N-1 pairwise interfaces from the first joined member to each subsequent one. KiCad merges them into one logical net by common pin.

In [ ]:
rail_3v3 = board.power("rail_3v3", voltage_v=3.3)
rail_3v3 += "buck_3v3.VOUT", "mcu.VDD", "imu.VDD"

gnd = board.gnd()
gnd += "buck_3v3.GND", "mcu.GND", "imu.GND"

sda, scl = board.i2c("bus0")
sda += "mcu.SDA", "imu.SDA"
scl += "mcu.SCL", "imu.SCL"

print(board.summary())
board

## Cell 6 — ERC + render + write `.kicad_sch`

`check_erc()` regenerates the .kicad_sch then runs `kicad-cli sch erc`. Raises `MultipleERCViolations` on any real violation. `show()` renders the full board inline. `write_kicad()` is the final artifact handed to eeschema for hand-tune.

In [ ]:
board.check_erc(expected_codes=(
    "pin_not_connected",        # engineer adds decoupling caps + pull-ups in eeschema
    "unconnected_wire_endpoint",
    "lib_symbol_issues",        # kicad-cli ignores project sym-lib-table — eeschema resolves fine
))
board.show()

In [ ]:
zip_path = board.export_kicad("control_hub_v1.zip", unzip=True)
print(f"phase-2 artifact → {zip_path}")
print(f"unpacked next to it  → {zip_path.with_suffix('')}/")
print("open the .kicad_pro in eeschema to hand-tune.")